Anchor Project: Decision Analytics for Supplier Quality Risk

Purpose of this notebook:
1) Load the raw recall dataset safely (encoding + messy rows)
2) Validate the 'classification' field
3) Keep ONLY true FDA severity classes: Class I / II / III
4) Map severity classes to a numeric severity score (5 / 3 / 1)
5) Save a clean processed datamset for downstream notebooks

Output:
data/processed/recalls_severity_clean.csv

In [1]:
import pandas as pd

In [2]:
# -----------------------------
# STEP 0: File paths
# -----------------------------
RAW_PATH = "data/raw/fda_drug_recalls.csv"
OUT_PATH = "data/processed/recalls_severity_clean.csv"

In [3]:
# -----------------------------
# STEP 1: Load raw data safely
# -----------------------------
df = pd.read_csv(
    RAW_PATH,
    encoding="latin1",
    engine="python",
    on_bad_lines="skip"
)

print("Raw dataset loaded")
print("Raw shape:", df.shape)
print("\nColumns:", list(df.columns))

df.head()

Raw dataset loaded
Raw shape: (641, 24)

Columns: ['country', 'city', 'address_1', 'reason_for_recall', 'address_2', 'product_quantity', 'code_info', 'center_classification_date', 'distribution_pattern', 'state', 'product_description', 'report_date', 'classification', 'openfda', 'recalling_firm', 'recall_number', 'initial_firm_notification', 'product_type', 'event_id', 'more_code_info', 'recall_initiation_date', 'postal_code', 'voluntary_mandated', 'status']


,country,city,address_1,reason_for_recall,address_2,product_quantity,code_info,center_classification_date,distribution_pattern,state,...,recalling_firm,recall_number,initial_firm_notification,product_type,event_id,more_code_info,recall_initiation_date,postal_code,voluntary_mandated,status
0,United States,Davie,4131 SW 47th Ave Ste 1403,Recall initiated as a precautionary measure du...,NaN,"1,990 bottles","UPC No. 632687615989; Lot No. 30661601, Exp. D...",20161025,"FL, MI, MS, and OH.",FL,...,Pharmatech LLC,F-0276-2017,Letter,Food,75272,NaN,20160808,33314-4036,Voluntary: Firm initiated,Ongoing
1,United States,Miami,13439 NW 19 LANE,Virginia State (VDACS) found Listeria monocyto...,NaN,144 pieces,UPC 635349 000390 Best By dates: 07/01/14 thr...,20141202,"FL, GA. NC, and TN",FL,...,"Oasis Brands, Inc",F-0609-2015,"Two or more of the following: Email, Fax, Lett...",Food,69516,20170328.0,20141010,33182,Voluntary: Firm initiated,Terminated
2,United States,Seattle,3429 Airport Way S,Coffee Toffee is recalled because pecan is lis...,NaN,24 packages,no codes,20180614,distributed in WA,WA,...,Yukon Jackson,F-1578-2018,Visit,Food,80233,20180625.0,20180525,98134-2139,Voluntary: Firm initiated,Terminated
3,United States,Brooklyn,47 Bridgewater St # 57,"Product contains dried peaches, but front labe...",NaN,unknown,UPC CODE: 6868978724496 BEST BEFORE: 11/15/2021,20200424,Unknown,NY,...,Rong Shing Trading NY Inc,F-0921-2020,"Two or more of the following: Email, Fax, Lett...",Food,85364,20210318.0,20200401,11222-3820,Voluntary: Firm initiated,Terminated
4,United States,Tipp City,320 N 2nd St,The firm stated that the product contains unde...,NaN,480/20 ib cases,"Product #29973B Code Dates: 10/20/2016, 11/8/...",20170605,Product was sent to one manufacturer in MI,OH,...,Trophy Nut Co Inc,F-2326-2017,Letter,Food,77213,20180213.0,20170505,45371,Voluntary: Firm initiated,Terminated


In [4]:
# -----------------------------
# STEP 2: Validate required columns
# -----------------------------
required_cols = ["classification", "reason_for_recall", "report_date", "recall_initiation_date"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}. Check your CSV columns and file.")

print("Required columns present:", required_cols)

Required columns present: ['classification', 'reason_for_recall', 'report_date', 'recall_initiation_date']


In [5]:
# -----------------------------
# STEP 3: Inspect classification values (raw)
# -----------------------------
print("--- Unique values in classification (raw) ---")
print(df["classification"].dropna().unique())

print("\n--- Top classification counts (raw) ---")
print(df["classification"].value_counts(dropna=False).head(20))

--- Unique values in classification (raw) ---
['Class II' 'Class I' 'Class III' 'Voluntary: Firm initiated'
 'FDA Mandated']

--- Top classification counts (raw) ---
classification
Class II                     268
Class I                      208
Voluntary: Firm initiated    137
Class III                     27
FDA Mandated                   1
Name: count, dtype: int64


In [6]:
# -----------------------------
# STEP 4: Filter to true severity classes only
# -----------------------------
# Public datasets can mix severity classes with other text (e.g., voluntary/mandated).
# For severity analysis we keep ONLY: Class I, Class II, Class III
valid_classes = ["Class I", "Class II", "Class III"]
df_severity = df[df["classification"].isin(valid_classes)].copy()

print("Filtered to severity classes only")
print("Severity-only shape:", df_severity.shape)

print("\n--- Severity distribution (Class I/II/III) ---")
print(df_severity["classification"].value_counts())

Filtered to severity classes only
Severity-only shape: (503, 24)

--- Severity distribution (Class I/II/III) ---
classification
Class II     268
Class I      208
Class III     27
Name: count, dtype: int64


In [7]:
# -----------------------------
# STEP 5: Map to numeric severity score
# -----------------------------
# Decision-analytics mapping (intentionally non-linear):
# Class I (high risk) -> 5
# Class II (moderate) -> 3
# Class III (low) -> 1
severity_map = {
    "Class I": 5,
    "Class II": 3,
    "Class III": 1
}

df_severity["severity_score"] = df_severity["classification"].map(severity_map)

# Sanity check: ensure no missing scores after mapping
missing_scores = df_severity["severity_score"].isna().sum()
if missing_scores != 0:
    raise ValueError(f"Found {missing_scores} rows with missing severity_score. Check mapping/data.")

df_severity[["classification", "severity_score"]].head()

,classification,severity_score
0,Class II,3
1,Class I,5
2,Class III,1
3,Class III,1
4,Class II,3


In [8]:
# -----------------------------
# STEP 6: Minimal cleaning of dates (optional, but professional)
# -----------------------------
# Convert date columns to datetime where possible
for col in ["report_date", "recall_initiation_date"]:
    df_severity[col] = pd.to_datetime(df_severity[col], errors="coerce")

print("Date parsing complete (invalid dates become NaT)")
print(df_severity[["report_date", "recall_initiation_date"]].head())

Date parsing complete (invalid dates become NaT)
  report_date recall_initiation_date
0  2016-11-02             2016-08-08
1  2014-12-10             2014-10-10
2  2018-06-20             2018-05-25
3  2020-05-06             2020-04-01
4  2017-06-14             2017-05-05


In [9]:
# -----------------------------
# STEP 7: Save processed dataset
# -----------------------------
df_severity.to_csv(OUT_PATH, index=False)

print(f"Saved processed severity dataset to: {OUT_PATH}")

Saved processed severity dataset to: data/processed/recalls_severity_clean.csv


In [10]:
# -----------------------------
# STEP 8: Quick summary you can quote
# -----------------------------
total = len(df_severity)
counts = df_severity["classification"].value_counts()
pct = (counts / total * 100).round(2)

summary = pd.DataFrame({"count": counts, "pct": pct})
print("--- Severity summary (cleaned) ---")
print(summary)

--- Severity summary (cleaned) ---
                count    pct
classification              
Class II          268  53.28
Class I           208  41.35
Class III          27   5.37
